# Analyse des Données - Impact de la Publicité sur les Ventes

## Introduction

Ce notebook analyse les données transactionnelles d'une marque et l'impact de ses campagnes publicitaires (TV et Programmatique) sur les ventes.

### Datasets disponibles:
1. **retailer.csv**: Données transactionnelles du site web
2. **tv_publisher.csv**: Données publicitaires TV
3. **programmatic_publisher.csv**: Données publicitaires programmatiques (bannières web)
4. **mapping_transac_publisher_tv.csv**: Table de mapping entre clients, devices et dsp_id
5. **socio_demo.csv**: Données socio-démographiques des clients

### Objectifs:
1. Comprendre le marché avec des KPIs clés
2. Analyser l'impact de la publicité sur les ventes


In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Bibliothèques importées avec succès")


## 1. Chargement des Données

Chargement de tous les datasets avec gestion optimisée de la mémoire.


In [ ]:
# Chemins des fichiers
DATA_PATH = 'input_data/X_2025/'

print("Chargement des données en cours...")
print("-" * 50)

# 1. Socio-démographie (petit fichier, chargement complet)
print("1/5 Chargement socio_demo.csv...")
df_socio = pd.read_csv(DATA_PATH + 'socio_demo.csv')
print(f"   → {len(df_socio):,} lignes chargées")

# 2. Mapping (table de correspondance)
print("2/5 Chargement mapping_transac_publisher_tv.csv...")
df_mapping = pd.read_csv(DATA_PATH + 'mapping_transac_publisher_tv.csv')
print(f"   → {len(df_mapping):,} lignes chargées")

# 3. Retailer (données transactionnelles)
print("3/5 Chargement retailer.csv...")
df_retailer = pd.read_csv(DATA_PATH + 'retailer.csv', 
                          parse_dates=['timestamp_utc'])
print(f"   → {len(df_retailer):,} lignes chargées")

# 4. TV Publisher
print("4/5 Chargement tv_publisher.csv...")
df_tv = pd.read_csv(DATA_PATH + 'tv_publisher.csv',
                    parse_dates=['timestamp_utc'])
print(f"   → {len(df_tv):,} lignes chargées")

# 5. Programmatic Publisher
print("5/5 Chargement programmatic_publisher.csv...")
df_prog = pd.read_csv(DATA_PATH + 'programmatic_publisher.csv',
                      parse_dates=['timestamp_utc'])
print(f"   → {len(df_prog):,} lignes chargées")

print("-" * 50)
print("✓ Toutes les données sont chargées !")


In [ ]:
# Aperçu rapide des données
print("APERÇU DES DONNÉES")
print("=" * 80)

datasets = {
    'Retailer': df_retailer,
    'TV Publisher': df_tv,
    'Programmatic': df_prog,
    'Mapping': df_mapping,
    'Socio-Demo': df_socio
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(f"  - Shape: {df.shape}")
    print(f"  - Colonnes: {list(df.columns)}")
    print(f"  - Mémoire: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


## 2. Exploration et Nettoyage des Données


In [ ]:
# Vérification des valeurs manquantes
print("VALEURS MANQUANTES")
print("=" * 80)

for name, df in datasets.items():
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n{name}:")
        print(missing[missing > 0])
    else:
        print(f"\n{name}: ✓ Aucune valeur manquante")


In [ ]:
# Statistiques descriptives - Retailer
print("STATISTIQUES DESCRIPTIVES - RETAILER")
print("=" * 80)
print(df_retailer.head(10))
print("\nTypes d'événements:")
print(df_retailer['event_name'].value_counts())
print("\nPériode couverte:")
print(f"  Du {df_retailer['timestamp_utc'].min()} au {df_retailer['timestamp_utc'].max()}")


## 3. Analyse du Marché - KPIs Clés

### 3.1 Vue d'ensemble des transactions


In [ ]:
# Filtrer uniquement les transactions (achats)
df_purchases = df_retailer[df_retailer['event_name'] == 'transaction'].copy()

print("VUE D'ENSEMBLE DES TRANSACTIONS")
print("=" * 80)
print(f"Nombre total de transactions: {len(df_purchases):,}")
print(f"Nombre de clients uniques: {df_purchases['customer_id'].nunique():,}")
print(f"Nombre de produits uniques: {df_purchases['product_name'].nunique():,}")
print(f"Nombre de marques: {df_purchases['brand'].nunique():,}")
print(f"\nChiffre d'affaires total: {df_purchases['sales'].sum():,.2f} $")
print(f"Quantité totale vendue: {df_purchases['quantity'].sum():,.0f} unités")


In [ ]:
# KPIs principaux
print("\nKPIS PRINCIPAUX")
print("=" * 80)

# Panier moyen
avg_basket = df_purchases['sales'].mean()
print(f"Panier moyen: {avg_basket:.2f} $")

# Panier moyen par acheteur (somme des achats / nombre de clients)
total_per_customer = df_purchases.groupby('customer_id')['sales'].sum()
avg_basket_per_buyer = total_per_customer.mean()
print(f"Panier moyen par acheteur (total dépenses): {avg_basket_per_buyer:.2f} $")

# Nombre moyen de transactions par client
transactions_per_customer = df_purchases.groupby('customer_id').size()
avg_transactions = transactions_per_customer.mean()
print(f"Nombre moyen de transactions par client: {avg_transactions:.2f}")

# Prix moyen par unité
avg_unit_price = df_purchases['sales'].sum() / df_purchases['quantity'].sum()
print(f"Prix moyen par unité: {avg_unit_price:.2f} $")

# Quantité moyenne par transaction
avg_quantity = df_purchases['quantity'].mean()
print(f"Quantité moyenne par transaction: {avg_quantity:.2f} unités")


### 3.2 Distribution des ventes


In [ ]:
# Graphique: Distribution du panier moyen
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogramme du montant des transactions
axes[0].hist(df_purchases['sales'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(avg_basket, color='red', linestyle='--', linewidth=2, label=f'Moyenne: {avg_basket:.2f}$')
axes[0].set_xlabel('Montant de la transaction ($)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des Montants de Transaction')
axes[0].legend()
axes[0].set_xlim(0, df_purchases['sales'].quantile(0.99))

# Boxplot par marque (top 10)
top_brands = df_purchases['brand'].value_counts().head(10).index
df_top_brands = df_purchases[df_purchases['brand'].isin(top_brands)]
df_top_brands.boxplot(column='sales', by='brand', ax=axes[1], rot=45)
axes[1].set_xlabel('Marque')
axes[1].set_ylabel('Montant ($)')
axes[1].set_title('Distribution des Ventes par Marque (Top 10)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print("✓ Graphiques générés")


### 3.3 Analyse temporelle des ventes


In [ ]:
# Evolution des ventes dans le temps
df_purchases['date'] = df_purchases['timestamp_utc'].dt.date
daily_sales = df_purchases.groupby('date').agg({
    'sales': 'sum',
    'customer_id': 'count'
}).reset_index()
daily_sales.columns = ['date', 'revenue', 'transactions']

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Chiffre d'affaires quotidien
axes[0].plot(daily_sales['date'], daily_sales['revenue'], linewidth=2)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Chiffre d\'affaires ($)')
axes[0].set_title('Évolution du Chiffre d\'Affaires Quotidien')
axes[0].grid(True, alpha=0.3)

# Nombre de transactions quotidiennes
axes[1].plot(daily_sales['date'], daily_sales['transactions'], linewidth=2, color='orange')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Nombre de transactions')
axes[1].set_title('Nombre de Transactions Quotidiennes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Analyse temporelle effectuée")


### 3.4 Analyse par marque et produit


In [ ]:
# Top 10 marques par chiffre d'affaires
brand_performance = df_purchases.groupby('brand').agg({
    'sales': 'sum',
    'quantity': 'sum',
    'customer_id': 'count'
}).reset_index()
brand_performance.columns = ['brand', 'revenue', 'quantity', 'transactions']
brand_performance = brand_performance.sort_values('revenue', ascending=False)

print("TOP 10 MARQUES PAR CHIFFRE D'AFFAIRES")
print("=" * 80)
print(brand_performance.head(10).to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top 10 marques par CA
top10_brands = brand_performance.head(10)
axes[0].barh(top10_brands['brand'], top10_brands['revenue'])
axes[0].set_xlabel('Chiffre d\'affaires ($)')
axes[0].set_title('Top 10 Marques par Chiffre d\'Affaires')
axes[0].invert_yaxis()

# Part de marché (top 10)
axes[1].pie(top10_brands['revenue'], labels=top10_brands['brand'], autopct='%1.1f%%')
axes[1].set_title('Part de Marché (Top 10)')

plt.tight_layout()
plt.show()


In [ ]:
# Top 15 produits
product_performance = df_purchases.groupby('product_name').agg({
    'sales': 'sum',
    'quantity': 'sum',
    'customer_id': 'count'
}).reset_index()
product_performance.columns = ['product', 'revenue', 'quantity', 'transactions']
product_performance = product_performance.sort_values('revenue', ascending=False)

print("\nTOP 15 PRODUITS PAR CHIFFRE D'AFFAIRES")
print("=" * 80)
print(product_performance.head(15).to_string(index=False))

# Visualisation
plt.figure(figsize=(12, 8))
top15_products = product_performance.head(15)
plt.barh(range(len(top15_products)), top15_products['revenue'])
plt.yticks(range(len(top15_products)), top15_products['product'])
plt.xlabel('Chiffre d\'affaires ($)')
plt.title('Top 15 Produits par Chiffre d\'Affaires')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
